# Dinosaur variant classifier (training)

Trains an image classifier for **four toy dinosaur variants** plus a **no dinosaur** class from folder-organized datasets.

**Moving camera:** the training pipeline uses strong augmentation (color jitter, blur, affine motion, flips) so the model generalizes when the scene and viewpoint change—not only when the toy moves.

## Dataset layout

Put images under one root folder, **one subfolder per class**. Names can be anything; folder names become class labels.

```
DATA_ROOT/
  variant_a/     # e.g. red spots
  variant_b/
  variant_c/
  variant_d/
  no_dinosaur/   # empty scene, clutter without toy, etc.
```

Supported formats: `.jpg`, `.jpeg`, `.png`, `.bmp`, `.webp` (anything `torchvision` reads).

## Google Colab

This notebook is written to run **locally or on Colab**.

1. **Runtime → Change runtime type → GPU** (recommended; CPU works but is slower).
2. Colab already includes **PyTorch**, **torchvision**, and **scikit-learn**; you usually do not need `pip install`.
3. **Dataset:** upload a zip via the file sidebar, unzip so class folders live under `/content/dataset`, or mount **Google Drive** and point `DATA_ROOT` at a folder on Drive (see the configuration cell).
4. **Saving outputs:** `/content` is cleared when the runtime disconnects. Set `OUTPUT_DIR` to a **Drive path** after mounting, or download `artifacts/` from the file browser before closing the session.

## Local setup

```bash
pip install -r requirements.txt
```

In [ ]:
# Google Colab — optional: mount Drive if your dataset or OUTPUT_DIR lives on Drive.
# Uncomment the next two lines, authenticate, then set DATA_ROOT / OUTPUT_DIR in the config cell.
# from google.colab import drive
# drive.mount("/content/drive")

pass

In [ ]:
from __future__ import annotations

import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

## Configuration

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# --- paths ---
if IN_COLAB:
    # Upload & unzip your dataset to /content/dataset, or after mounting Drive use e.g.:
    # DATA_ROOT = Path("/content/drive/MyDrive/your_project/dataset")
    DATA_ROOT = Path("/content/dataset")
    # Ephemeral unless you switch to Drive (recommended before disconnecting):
    # OUTPUT_DIR = Path("/content/drive/MyDrive/your_project/artifacts")
    OUTPUT_DIR = Path("/content/artifacts")
else:
    DATA_ROOT = Path("data/dataset")
    OUTPUT_DIR = Path("artifacts")

MODEL_PATH = OUTPUT_DIR / "dinosaur_classifier.pt"
CLASSES_JSON = OUTPUT_DIR / "class_names.json"

# --- training ---
IMAGE_SIZE = 224          # ResNet input
BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.2
# Colab (Linux): a few workers speed up loading. Use 0 if DataLoader errors or hangs.
NUM_WORKERS = min(4, os.cpu_count() or 2) if IN_COLAB else 0
SEED = 42

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Colab:", IN_COLAB)
print("Device:", DEVICE)
print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("DataLoader workers:", NUM_WORKERS)

## Unzip dataset (optional)

If your data is in **one zip** whose contents are class folders (or one outer folder that contains those class folders), set `DATASET_ZIP` in the cell below and run it. It sets `DATA_ROOT` to the correct folder for `ImageFolder`.

If you use **several zips** (e.g. one per class), unzip them yourself so you have `DATA_ROOT/variant_a/`, `DATA_ROOT/variant_b/`, … then leave `DATASET_ZIP = None` and point `DATA_ROOT` at that folder in the configuration cell.

In [ ]:
import shutil
import zipfile

# Path to a single archive, or None if DATA_ROOT already points to an unzipped folder
DATASET_ZIP = None  # e.g. Path("/content/dataset.zip") or Path("/content/drive/MyDrive/project/dataset.zip")


def infer_imagefolder_root(extract_to: Path) -> Path:
    """After extractall: if the zip had one top-level folder that wraps the class folders, return that folder; else return extract_to."""
    extract_to = extract_to.resolve()
    items = [
        p
        for p in extract_to.iterdir()
        if p.is_dir() and p.name not in ("__MACOSX",) and not p.name.startswith(".")
    ]
    files_at_root = [p for p in extract_to.iterdir() if p.is_file()]
    if len(items) == 1 and not files_at_root:
        inner = items[0]
        child_dirs = [p for p in inner.iterdir() if p.is_dir() and p.name != "__MACOSX"]
        if len(child_dirs) >= 1:
            return inner
    return extract_to


if DATASET_ZIP is not None:
    zip_path = Path(DATASET_ZIP).expanduser().resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(f"Zip not found: {zip_path}")
    staging = (
        Path("/content/_dataset_from_zip")
        if IN_COLAB
        else Path("_dataset_from_zip")
    )
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(staging)
    DATA_ROOT = infer_imagefolder_root(staging)
    print("Unzipped from:", zip_path)
    print("Using DATA_ROOT:", DATA_ROOT.resolve())
else:
    print("DATASET_ZIP is None — using DATA_ROOT from the configuration cell:", DATA_ROOT.resolve())

## Load data and split train / validation

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_ROOT.resolve()}\n"
        "Create it and add one subfolder per class (see notebook intro)."
    )

# Base transform: resize for decoding; augmentation applied separately to train
train_tfms = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=25, translate=(0.1, 0.1), scale=(0.85, 1.15)),
        transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.08),
        transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.25),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_tfms = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# PIL images so Resize / augmentation work as intended
full_ds = datasets.ImageFolder(root=str(DATA_ROOT))
class_names = full_ds.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)

if num_classes < 2:
    raise ValueError("Need at least 2 class folders under DATA_ROOT.")

n_total = len(full_ds)
n_val = int(n_total * VAL_FRACTION)
n_train = n_total - n_val
if n_train < 1 or n_val < 1:
    raise ValueError("Not enough images for train/val split; add more images per class.")

gen = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(full_ds, [n_train, n_val], generator=gen)


class TransformSubset(torch.utils.data.Dataset):
    """Apply different transforms to train vs val subsets."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, y = self.subset[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, y


train_ds = TransformSubset(train_subset, train_tfms)
val_ds = TransformSubset(val_subset, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

## Model: ResNet-18 + ImageNet head swap

Small, fast, and strong enough for cropped or full-frame toy classification when you have enough labeled images per class.

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    m = models.resnet18(weights=weights)
    in_f = m.fc.in_features
    m.fc = nn.Linear(in_f, num_classes)
    return m


model = build_model(num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## Train and validate

In [ ]:
def run_epoch(model, loader, train: bool) -> tuple[float, float]:
    if train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            if train:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val = -1.0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    tl, ta = run_epoch(model, train_loader, train=True)
    vl, va = run_epoch(model, val_loader, train=False)
    scheduler.step()
    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)
    if va > best_val:
        best_val = va
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  train_loss={tl:.4f} acc={ta:.3f}  val_loss={vl:.4f} acc={va:.3f}")

if best_state is not None:
    model.load_state_dict(best_state)
print(f"Best val accuracy: {best_val:.4f}")

## Curves and confusion matrix

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss")
ax[0].legend()
ax[1].plot(history["train_acc"], label="train")
ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
all_y, all_pred = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        all_pred.append(pred)
        all_y.append(y.numpy())
y_true = np.concatenate(all_y)
y_pred = np.concatenate(all_pred)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_ylabel("True")
ax.set_xlabel("Predicted")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black", fontsize=9)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Save weights and class index map

At inference time, load `dinosaur_classifier.pt` and `class_names.json` so labels match training order (`ImageFolder` sorts folder names alphabetically).

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_classes": num_classes,
        "image_size": IMAGE_SIZE,
        "arch": "resnet18",
    },
    MODEL_PATH,
)
with open(CLASSES_JSON, "w", encoding="utf-8") as f:
    json.dump({"class_names": class_names}, f, indent=2)
print("Saved:", MODEL_PATH.resolve())
print("Saved:", CLASSES_JSON.resolve())

## Optional: quick sanity check on one batch

Denormalize and show a few validation images with predicted labels.

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def denormalize(t: torch.Tensor) -> torch.Tensor:
    return (t.cpu() * std + mean).clamp(0, 1)


model.eval()
x_b, y_b = next(iter(val_loader))
with torch.no_grad():
    pred = model(x_b.to(DEVICE)).argmax(dim=1).cpu()
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.ravel()
for i in range(min(8, x_b.size(0))):
    img = denormalize(x_b[i]).permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"t:{class_names[y_b[i]]}\np:{class_names[pred[i]]}", fontsize=8)
    axes[i].axis("off")
for j in range(i + 1, 8):
    axes[j].axis("off")
plt.suptitle("true (t) vs predicted (p)")
plt.tight_layout()
plt.show()